# Explore the macroalgal microbiome use case

This notebook analyzes MAGs for the macroalgal microbiome use case.

**Inputs**: data in `../data/marine-use-case/data/`:
- `drep.csv`: dRep clustering output
- `checkm2.tsv`: CheckM2 quality assessment
- `gtdb.tsv`: GTDB taxonomy classification
- `quast.tsv`: QUAST assembly statistics
- `bakta.tsv`: BAKTA genome annotation

**Outputs**
- Plots: `../results/marine-use-case/`

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

sys.path.insert(0, str((Path.cwd() / "bin").resolve()))
sys.path.insert(0, str(Path.cwd().resolve()))

from helpers import (
    load_dfs,
    compute_print_stats,
    explore_species_level_clusters_all,
    compute_taxo_classification_summary,
    get_all_taxo_levels,
    get_bakta_annot_df,
    get_kegg_path_df,
    get_relative_abund_taxo_levels,
    print_stats,
)

In [2]:
data_dp = Path("../data/use-cases/marine-microbiome/")
data_dp.exists()

data_dp_2 = Path("../data/marine-use-case/")
data_dp_2.exists()

result_dp = Path("../results/marine-use-case/")
result_dp.exists()

metadata_df, reps_df, coverage_df = load_dfs(data_dp_2, result_dp)

In [3]:
# WARNING: high percentage of unmapped
print("Percentage of unmapped reads for coverage")
print_stats(coverage_df.query("Genome == 'unmapped'").drop(columns="Genome").T.describe())

Percentage of unmapped reads for coverage
585: 54.92 ± 16.42, Median: 50.71, IQR: 45.86-61.87, Range: 41.01-73.04


# Quality

In [4]:
print(f"Total number of MAGs: {reps_df['Cluster members'].sum()}")
print(f"Total number of species-level clusters: {reps_df.shape[0]}")

Total number of MAGs: 628
Total number of species-level clusters: 585


In [5]:
explore_species_level_clusters_all(reps_df)

Species-level clusters with no contamination threshold
Total number: 585.0
Cluster members: 1.07 ± 0.30, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 1.18 ± 1.84, Median: 0.31, IQR: 0.03-1.54, Range: 0.00-10.18
Completeness: 36.19 ± 35.44, Median: 15.60, IQR: 7.90-69.00, Range: 0.00-100.00
Total length: 1.49 ± 1.51, Median: 0.73, IQR: 0.33-2.46, Range: 0.05-7.59

Species-level clusters with contamination < 5%
Total number: 549.0
Cluster members: 1.07 ± 0.30, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 0.80 ± 1.11, Median: 0.25, IQR: 0.02-1.21, Range: 0.00-4.88
Completeness: 34.96 ± 35.53, Median: 13.90, IQR: 7.70-68.10, Range: 2.40-100.00
Total length: 1.45 ± 1.48, Median: 0.68, IQR: 0.33-2.42, Range: 0.05-7.12

Species-level clusters with contamination < 10%
Total number: 584.0
Cluster members: 1.07 ± 0.30, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 1.16 ± 1.81, Median: 0.31, IQR: 0.03-1.54, Range: 0.00-9.98
Completeness: 36.13 ± 3

# Clusters given Bowers et al / MIMAG classification

## HQ: High-quality species-level clusters (contamination < 5% and completeness > 90%)

In [6]:
hq_df = reps_df.query("Contamination < 5 and Completeness > 90")
hq_df

,MAG,Domain,Phylum,Class,Order,Family,Genus,Species,Cluster members,Completeness,...,kegg_dTDP-D-angolosamine biosynthesis,kegg_dTDP-D-desosamine biosynthesis,kegg_dTDP-D-forosamine biosynthesis,kegg_dTDP-D-mycaminose biosynthesis,kegg_dTDP-L-megosamine biosynthesis,kegg_dTDP-L-mycarose biosynthesis,kegg_dTDP-L-oleandrose biosynthesis,kegg_dTDP-L-olivose biosynthesis,kegg_dTDP-L-rhamnose biosynthesis,kegg_dTDP-beta-L-noviose biosynthesis
0,SRR22878281_binette_bin2,Bacteria,Pseudomonadota,Alphaproteobacteria,Rhizobiales,Hyphomicrobiaceae,CAJQQK01,unclassified,1,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,NaN
1,SRR22878283_binette_bin11,Bacteria,Pseudomonadota,Gammaproteobacteria,Granulosicoccales,Granulosicoccaceae,CAKZSB01,unclassified,1,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66.67,NaN
2,SRR22878281_binette_bin38,Bacteria,Pseudomonadota,Gammaproteobacteria,Pseudomonadales,Azotimanducaceae,CALHVX01,unclassified,1,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,NaN
3,SRR22878281_binette_bin39,Bacteria,Pseudomonadota,Alphaproteobacteria,Caulobacterales,Maricaulaceae,Hellea,unclassified,1,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,NaN
4,SRR22878281_binette_bin26,Bacteria,Pseudomonadota,Alphaproteobacteria,Rhizobiales,Rhizobiaceae,Lentilitoribacter,Lentilitoribacter sp020628775,1,100.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.33,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,SRR22878281_binette_bin41,Bacteria,Verrucomicrobiota,Verrucomicrobiia,Verrucomicrobiales,DEV007,JAUIGD01,unclassified,1,90.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.33,NaN
98,SRR22878281_binette_bin31,Bacteria,Pseudomonadota,Alphaproteobacteria,Caulobacterales,Maricaulaceae,Robiginitomaculum_A,unclassified,1,90.7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,NaN
99,SRR22878281_binette_bin45,Bacteria,Pseudomonadota,Alphaproteobacteria,Caulobacterales,Maricaulaceae,JACOMS01,unclassified,1,90.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,NaN
100,SRR22878283_binette_bin36,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,Flavobacteriaceae,Croceitalea,Croceitalea vernalis,2,90.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,NaN


In [7]:
compute_print_stats(hq_df) 

Total number: 98.0
Cluster members: 1.23 ± 0.53, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 1.19 ± 1.07, Median: 1.02, IQR: 0.28-1.75, Range: 0.00-4.65
Completeness: 96.29 ± 3.26, Median: 96.90, IQR: 93.55-99.60, Range: 90.10-100.00
Total length: 3.75 ± 1.14, Median: 3.55, IQR: 2.96-4.29, Range: 1.03-7.12


### Taxonomy

In [8]:
compute_taxo_classification_summary(hq_df)

,Unclassified clusters,Classified clusters,Unclassified clusters %,Classified clusters %
Domain,0,98,0.0,100.0
Phylum,0,98,0.0,100.0
Class,0,98,0.0,100.0
Order,0,98,0.0,100.0
Family,3,95,3.06,96.94
Genus,12,86,12.24,87.76
Species,84,14,85.71,14.29


In [9]:
hq_taxo_levels = get_all_taxo_levels(hq_df)


Level: Domain


,Cluster,Cluster %,Total MAG count
Domain,,,
Bacteria,98.0,100.0,121.0
TOTAL,98.0,100.0,121.0



Level: Phylum


,Cluster,Cluster %,Total MAG count
Phylum,,,
Pseudomonadota,56.0,57.142857,62.0
Bacteroidota,23.0,23.469388,38.0
Cyanobacteriota,5.0,5.102041,5.0
Actinomycetota,3.0,3.061224,3.0
Myxococcota_A,3.0,3.061224,3.0
Bdellovibrionota,2.0,2.040816,3.0
Patescibacteriota,2.0,2.040816,3.0
Acidobacteriota,1.0,1.020408,1.0
Bdellovibrionota_B,1.0,1.020408,1.0



Level: Class


...

,Cluster,Cluster %,Total MAG count
Family,,,
Flavobacteriaceae,20.0,20.408163,35.0
Alteromonadaceae,5.0,5.102041,9.0
Cellvibrionaceae,7.0,7.142857,8.0
Maricaulaceae,7.0,7.142857,7.0
Sphingomonadaceae,6.0,6.122449,6.0
Granulosicoccaceae,6.0,6.122449,6.0
Xenococcaceae,4.0,4.081633,4.0
Arenicellaceae,4.0,4.081633,4.0
Geminicoccaceae,3.0,3.061224,3.0



Level: Genus


,Cluster,Cluster %,Total MAG count
Genus,,,
unclassified,12.0,12.244898,13.0
Aquimarina,5.0,5.102041,6.0
Pseudoalteromonas,2.0,2.040816,5.0
Postechiella,2.0,2.040816,5.0
Marinagarivorans,3.0,3.061224,4.0
...,...,...,...
Aridibacter,1.0,1.020408,1.0
Litorimonas,1.0,1.020408,1.0
CALHVX01,1.0,1.020408,1.0



Level: Species


,Cluster,Cluster %,Total MAG count
Species,,,
unclassified,84.0,85.714286,94.0
Cellulophaga lytica,1.0,1.020408,3.0
Dokdonia sp947496725,1.0,1.020408,3.0
Olleya sediminilitoris,1.0,1.020408,3.0
Pseudoalteromonas marina,1.0,1.020408,3.0
Croceitalea vernalis,1.0,1.020408,2.0
Lacinutrix sp000211855,1.0,1.020408,2.0
Maribacter litoralis,1.0,1.020408,2.0
Marinagarivorans sp947494095,1.0,1.020408,2.0


### Relative abundance

In [10]:
hq_relative_abund_df = get_relative_abund_taxo_levels(hq_df, coverage_df)

Unmapped reads: 88.25 ± 5.19, Median: 87.00, IQR: 85.40-90.47, Range: 83.79-93.94
Mapped reads: 11.75 ± 5.19, Median: 13.00, IQR: 9.53-14.60, Range: 6.06-16.21

Level: Family


,count,mean,std,min,25%,50%,75%,max
Family,,,,,,,,
Flavobacteriaceae,3.0,52.156285,9.624523,41.043810,49.314675,57.585541,57.712523,57.839504
Cellvibrionaceae,3.0,11.247012,14.027803,0.460200,3.317891,6.175582,16.640417,27.105253
Alteromonadaceae,3.0,6.182130,4.482648,2.728903,3.649201,4.569498,7.908743,11.247988
Sphingomonadaceae,3.0,4.582422,3.959566,0.251749,2.864909,5.478069,6.747759,8.017449
Maricaulaceae,3.0,2.995427,2.753324,0.342388,1.573577,2.804766,4.321946,5.839126
Granulosicoccaceae,3.0,2.397838,2.023309,0.316448,1.417969,2.519489,3.438533,4.357576
Xenococcaceae,3.0,2.158519,2.416898,0.037409,0.842897,1.648386,3.219075,4.789763
Arenicellaceae,3.0,1.979363,1.715131,0.202951,1.156135,2.109318,2.867570,3.625821
Hyphomonadaceae,3.0,1.810630,2.568614,0.070348,0.335566,0.600784,2.680771,4.760758



Level: Genus


,count,mean,std,min,25%,50%,75%,max
Genus,,,,,,,,
Cellulophaga,3.0,14.788066,19.237199,0.576587,3.842713,7.108840,21.893806,36.678772
Aquimarina,3.0,14.658679,22.439511,0.141580,1.735966,3.330351,21.917229,40.504107
unclassified,3.0,14.165283,17.804443,3.793401,3.886022,3.978644,19.351224,34.723805
Dokdonia,3.0,6.290110,6.474611,1.926067,2.570557,3.215047,8.472131,13.729215
Postechiella,3.0,5.368177,5.232469,0.486312,2.606260,4.726209,7.809109,10.892010
...,...,...,...,...,...,...,...,...
Psychrobacter,3.0,0.171743,0.297468,0.000000,0.000000,0.000000,0.257615,0.515230
CAKZUP01,3.0,0.140389,0.210904,0.000000,0.019125,0.038250,0.210583,0.382916
UBA4427,3.0,0.139303,0.241280,0.000000,0.000000,0.000000,0.208955,0.417910



Level: Species


,count,mean,std,min,25%,50%,75%,max
Species,,,,,,,,
unclassified,3.0,67.180552,25.343968,40.905096,55.032328,69.159561,80.318280,91.476999
Cellulophaga lytica,3.0,14.788066,19.237199,0.576587,3.842713,7.108840,21.893806,36.678772
Dokdonia sp947496725,3.0,6.290110,6.474611,1.926067,2.570557,3.215047,8.472131,13.729215
Pseudoalteromonas atlantica,3.0,2.269610,2.123595,0.302196,1.143994,1.985793,3.253317,4.520842
Lacinutrix sp000211855,3.0,2.171252,1.950391,0.247613,1.183210,2.118807,3.133072,4.147337
Maribacter litoralis,3.0,1.820617,2.333411,0.166246,0.486141,0.806037,2.647802,4.489567
Pseudoalteromonas marina,3.0,1.518409,1.574863,0.578385,0.609337,0.640289,1.988421,3.336553
Marinagarivorans sp947494095,3.0,0.759595,0.946660,0.035878,0.223933,0.411989,1.121454,1.830919
Croceitalea vernalis,3.0,0.688653,0.482567,0.165586,0.474713,0.783841,0.950187,1.116533


### Functions

In [11]:
print_stats(get_bakta_annot_df(hq_df).describe())

CDSs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
CRISPR arrays: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
gaps: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
hypotheticals: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
ncRNA regions: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
ncRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriCs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriTs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriVs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
pseudogenes: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
rRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
sORFs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
signal peptides: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
tRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
tmRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan


In [12]:
get_kegg_path_df(hq_df)

Before removing rows and columns with only zeros:
Clusters: 98
KEGG modules: 405

After removing rows and columns with only zeros:
Clusters: 98
KEGG modules: 378

KEGG modules: 194.52 ± 21.38, Median: 195.00, IQR: 186.25-204.75, Range: 78.00-245.00


,"10-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 10-membered enediyne core",3-Hydroxypropionate bi-cycle,"9-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 9-membered enediyne core",ADP-L-glycero-D-manno-heptose biosynthesis,"Abscisic acid biosynthesis, beta-carotene => abscisic acid","Acarbose biosynthesis, sedoheptulopyranose-7P => acarbose",Acylglycerol degradation,"Adenine ribonucleotide biosynthesis, IMP => ADP,ATP","Adenine ribonucleotide degradation, AMP => Urate","Aerobactin biosynthesis, lysine => aerobactin",...,beta-Oxidation,"beta-Oxidation, acyl-CoA synthesis","beta-Oxidation, peroxisome, VLCFA","beta-Oxidation, peroxisome, tri/dihydroxycholestanoyl-CoA => choloyl/chenodeoxycholoyl-CoA",dTDP-D-angolosamine biosynthesis,dTDP-D-desosamine biosynthesis,dTDP-D-forosamine biosynthesis,dTDP-L-megosamine biosynthesis,dTDP-L-olivose biosynthesis,dTDP-L-rhamnose biosynthesis
0,0.0,52.78,0.0,0.0,0.0,0.0,0.0,100.0,77.78,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.00
1,0.0,22.22,0.0,20.0,0.0,12.5,0.0,100.0,100.00,0.0,...,100.0,100.0,33.33,0.0,0.0,0.0,0.0,0.0,0.0,66.67
2,0.0,62.50,0.0,60.0,0.0,0.0,0.0,100.0,77.78,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.00
3,0.0,22.22,0.0,0.0,20.0,0.0,0.0,75.0,66.67,0.0,...,100.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.00
4,0.0,27.78,0.0,0.0,0.0,0.0,0.0,75.0,100.00,0.0,...,50.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,33.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,0.0,12.50,0.0,100.0,0.0,0.0,50.0,75.0,83.33,0.0,...,0.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,83.33
98,0.0,27.78,0.0,0.0,0.0,0.0,0.0,100.0,66.67,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.00
99,0.0,25.00,0.0,0.0,0.0,0.0,50.0,100.0,66.67,0.0,...,100.0,100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,100.00
100,0.0,27.78,0.0,0.0,20.0,0.0,0.0,100.0,50.00,0.0,...,100.0,100.0,33.33,0.0,0.0,0.0,0.0,0.0,0.0,100.00


## MQ: Medium-quality species-level clusters (contamination < 10% and completeness > 50%)


In [13]:
mq_df = reps_df.query("Contamination < 10 and Completeness > 50")
compute_print_stats(mq_df) 

Total number: 179.0
Cluster members: 1.20 ± 0.47, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-3.00
Contamination: 2.22 ± 2.12, Median: 1.64, IQR: 0.79-2.88, Range: 0.00-9.98
Completeness: 86.29 ± 14.17, Median: 91.90, IQR: 79.65-97.50, Range: 50.60-100.00
Total length: 3.33 ± 1.21, Median: 3.18, IQR: 2.47-3.98, Range: 0.22-7.12


### Taxonomy

In [14]:
compute_taxo_classification_summary(mq_df)

,Unclassified clusters,Classified clusters,Unclassified clusters %,Classified clusters %
Domain,2,177,1.12,98.88
Phylum,3,176,1.68,98.32
Class,3,176,1.68,98.32
Order,3,176,1.68,98.32
Family,6,173,3.35,96.65
Genus,22,157,12.29,87.71
Species,152,27,84.92,15.08


In [15]:
mq_taxo_levels = get_all_taxo_levels(mq_df)


Level: Domain


,Cluster,Cluster %,Total MAG count
Domain,,,
Bacteria,176.0,98.324022,210.0
unclassified,2.0,1.117318,3.0
Archaea,1.0,0.558659,1.0
TOTAL,179.0,100.000000,214.0



Level: Phylum


,Cluster,Cluster %,Total MAG count
Phylum,,,
Pseudomonadota,89.0,49.720670,95.0
Bacteroidota,44.0,24.581006,66.0
Actinomycetota,10.0,5.586592,12.0
Cyanobacteriota,8.0,4.469274,8.0
Myxococcota_A,5.0,2.793296,5.0
Verrucomicrobiota,5.0,2.793296,5.0
unclassified,3.0,1.675978,5.0
Bdellovibrionota,3.0,1.675978,4.0
Patescibacteriota,3.0,1.675978,4.0



Level: Class


...

,Cluster,Cluster %,Total MAG count
Family,,,
Flavobacteriaceae,40.0,22.346369,61.0
Alteromonadaceae,6.0,3.351955,10.0
Maricaulaceae,10.0,5.586592,10.0
Rhodobacteraceae,10.0,5.586592,10.0
Cellvibrionaceae,8.0,4.469274,9.0
unclassified,6.0,3.351955,8.0
Granulosicoccaceae,8.0,4.469274,8.0
Sphingomonadaceae,8.0,4.469274,8.0
Arenicellaceae,7.0,3.910615,7.0



Level: Genus


,Cluster,Cluster %,Total MAG count
Genus,,,
unclassified,22.0,12.290503,25.0
Aquimarina,7.0,3.910615,8.0
Arenicella,7.0,3.910615,7.0
JAALLB01,6.0,3.351955,6.0
SHLQ01,4.0,2.234637,6.0
...,...,...,...
Aureisphaera,1.0,0.558659,1.0
CAKZSB01,1.0,0.558659,1.0
CAKZUP01,1.0,0.558659,1.0



Level: Species


,Cluster,Cluster %,Total MAG count
Species,,,
unclassified,152.0,84.916201,171.0
Cellulophaga lytica,1.0,0.558659,3.0
Dokdonia sp947496725,1.0,0.558659,3.0
Pseudoalteromonas marina,1.0,0.558659,3.0
Olleya sediminilitoris,1.0,0.558659,3.0
Polaribacter marinaquae,1.0,0.558659,2.0
Croceitalea vernalis,1.0,0.558659,2.0
Lacinutrix sp000211855,1.0,0.558659,2.0
Maribacter litoralis,1.0,0.558659,2.0


### Relative abundance

In [16]:
mq_relative_abund_df = get_relative_abund_taxo_levels(mq_df, coverage_df)

Unmapped reads: 83.75 ± 5.79, Median: 83.42, IQR: 80.77-86.55, Range: 78.13-89.69
Mapped reads: 16.25 ± 5.79, Median: 16.58, IQR: 13.45-19.23, Range: 10.31-21.87

Level: Family


,count,mean,std,min,25%,50%,75%,max
Family,,,,,,,,
Flavobacteriaceae,3.0,46.903582,7.483990,38.877325,43.510068,48.142810,50.916710,53.690610
Cellvibrionaceae,3.0,7.126049,7.984414,0.534468,2.686910,4.839353,10.421840,16.004327
Alteromonadaceae,3.0,4.269631,2.363702,2.031720,3.033599,4.035479,5.388586,6.741694
unclassified,3.0,3.896944,1.662662,2.668607,2.950947,3.233287,4.511112,5.788937
Sphingomonadaceae,3.0,3.778945,3.335278,0.178455,2.286912,4.395369,5.579190,6.763012
Maricaulaceae,3.0,3.489539,2.795352,0.631244,2.125618,3.619991,4.918686,6.217380
Vibrionaceae,3.0,3.405810,4.887230,0.077837,0.600339,1.122840,5.069797,9.016753
Arenicellaceae,3.0,2.986118,1.261920,1.865884,2.302559,2.739235,3.546235,4.353235
Granulosicoccaceae,3.0,2.133295,1.389479,0.751365,1.434843,2.118321,2.824261,3.530201



Level: Genus


,count,mean,std,min,25%,50%,75%,max
Genus,,,,,,,,
unclassified,3.0,13.393447,13.005598,5.402363,5.889940,6.377517,17.388989,28.400460
Aquimarina,3.0,11.634631,17.745180,0.677739,1.397887,2.118035,17.113077,32.108120
Cellulophaga,3.0,10.604697,14.481026,0.451829,2.313446,4.175063,15.681131,27.187198
Dokdonia,3.0,4.841993,5.328302,1.586932,1.767462,1.947991,6.469523,10.991054
Postechiella,3.0,4.440098,4.447361,0.383215,2.062490,3.741766,6.468540,9.195314
...,...,...,...,...,...,...,...,...
JAUVQU01,3.0,0.092870,0.096903,0.000000,0.042627,0.085253,0.139305,0.193356
JAUIGD01,3.0,0.088607,0.153471,0.000000,0.000000,0.000000,0.132910,0.265820
CALGNO01,3.0,0.068183,0.118097,0.000000,0.000000,0.000000,0.102275,0.204550



Level: Species


,count,mean,std,min,25%,50%,75%,max
Species,,,,,,,,
unclassified,3.0,69.123444,19.241865,50.257775,59.324844,68.391914,78.556278,88.720642
Cellulophaga lytica,3.0,10.604697,14.481026,0.451829,2.313446,4.175063,15.681131,27.187198
Dokdonia sp947496725,3.0,4.524654,4.898245,1.509318,1.698767,1.888216,6.032322,10.176428
Vibrio cyclitrophicus,3.0,1.919247,2.799828,0.045275,0.310008,0.574741,2.856234,5.137726
Tenacibaculum sp004337695,3.0,1.642492,1.647368,0.393574,0.708979,1.024385,2.266952,3.509519
Vibrio coralliirubri,3.0,1.486563,2.087908,0.032562,0.290331,0.548099,2.213563,3.879027
Pseudoalteromonas atlantica,3.0,1.454614,1.209247,0.236809,0.854363,1.471918,2.063517,2.655117
Lacinutrix sp000211855,3.0,1.400101,1.130534,0.194036,0.882274,1.570511,2.003133,2.435756
Maribacter litoralis,3.0,1.121493,1.332879,0.130275,0.363864,0.597454,1.617102,2.636749


### Functions

In [17]:
print_stats(get_bakta_annot_df(mq_df).describe())

CDSs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
CRISPR arrays: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
gaps: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
hypotheticals: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
ncRNA regions: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
ncRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriCs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriTs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
oriVs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
pseudogenes: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
rRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
sORFs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
signal peptides: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
tRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan
tmRNAs: nan ± nan, Median: nan, IQR: nan-nan, Range: nan-nan


In [18]:
get_kegg_path_df(mq_df)

Before removing rows and columns with only zeros:
Clusters: 179
KEGG modules: 405

After removing rows and columns with only zeros:
Clusters: 179
KEGG modules: 386

KEGG modules: 187.21 ± 32.30, Median: 191.00, IQR: 181.00-201.00, Range: 2.00-245.00


,"10-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 10-membered enediyne core","2-Oxocarboxylic acid chain extension, 2-oxoglutarate => 2-oxoadipate => 2-oxopimelate => 2-oxosuberate",3-Hydroxypropionate bi-cycle,"9-membered enediyne core biosynthesis, malonyl-CoA => 3-hydroxyhexadeca-4,6,8,10,12,14-hexaenoyl-ACP => 9-membered enediyne core",ADP-L-glycero-D-manno-heptose biosynthesis,"Abscisic acid biosynthesis, beta-carotene => abscisic acid","Acarbose biosynthesis, sedoheptulopyranose-7P => acarbose",Acylglycerol degradation,"Adenine ribonucleotide biosynthesis, IMP => ADP,ATP","Adenine ribonucleotide degradation, AMP => Urate",...,"beta-Oxidation, peroxisome, tri/dihydroxycholestanoyl-CoA => choloyl/chenodeoxycholoyl-CoA",dTDP-D-angolosamine biosynthesis,dTDP-D-desosamine biosynthesis,dTDP-D-forosamine biosynthesis,dTDP-D-mycaminose biosynthesis,dTDP-L-megosamine biosynthesis,dTDP-L-mycarose biosynthesis,dTDP-L-olivose biosynthesis,dTDP-L-rhamnose biosynthesis,dTDP-beta-L-noviose biosynthesis
0,0.0,0.0,52.78,0.0,0.0,0.0,0.0,0.0,100.0,77.78,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,100.00,0.0
1,0.0,0.0,22.22,0.0,20.0,0.0,12.5,0.0,100.0,100.00,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,66.67,0.0
2,0.0,0.0,62.50,0.0,60.0,0.0,0.0,0.0,100.0,77.78,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,100.00,0.0
3,0.0,0.0,22.22,0.0,0.0,20.0,0.0,0.0,75.0,66.67,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,100.00,0.0
4,0.0,0.0,27.78,0.0,0.0,0.0,0.0,0.0,75.0,100.00,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,33.33,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,0.0,0.0,43.06,0.0,0.0,0.0,0.0,0.0,75.0,66.67,...,0.0,0.0,0.0,20.0,33.33,0.0,0.0,25.0,100.00,0.0
176,0.0,0.0,13.89,0.0,0.0,0.0,0.0,50.0,25.0,33.33,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,33.33,0.0
177,0.0,0.0,13.89,0.0,0.0,0.0,0.0,0.0,50.0,66.67,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,50.00,0.0
178,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0


### LQ: Low-quality species-level clusters (contamination < 10% and completeness < 50%)

In [19]:
lq_df = reps_df.query("Contamination < 10 and Completeness < 50")
compute_print_stats(lq_df)

Total number: 405.0
Cluster members: 1.02 ± 0.14, Median: 1.00, IQR: 1.00-1.00, Range: 1.00-2.00
Contamination: 0.70 ± 1.42, Median: 0.11, IQR: 0.01-0.61, Range: 0.00-8.14
Completeness: 13.97 ± 10.81, Median: 9.70, IQR: 6.80-17.00, Range: 0.00-49.40
Total length: 0.68 ± 0.69, Median: 0.44, IQR: 0.28-0.79, Range: 0.05-7.59
